In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def SOA_InP_auto_coerente(E_ingresso_filtrato, lambda_vett, guadagno_dB=20.0):
    
    potenze_in = np.abs(E_ingresso_filtrato) ** 2
    max_potenza = np.max(potenze_in)
    media_potenza = np.mean(potenze_in)
    
    if media_potenza == 0:
        media_potenza = 1e-12

    # Calcolo della potenza totale attualmente circolante nella cavità
    P_totale_cavita = np.sum(potenze_in)
    
    P_sat = 1
    
    #Calolo kcoerente
    rapporto_purezza = max_potenza / media_potenza
    soglia_inizio_laser = 3.0
    soglia_laser_pieno = 25.0
    
    if rapporto_purezza <= soglia_inizio_laser:
        k_coer = 0.0
    elif rapporto_purezza >= soglia_laser_pieno:
        k_coer = 1.0
    else:
        k_coer = (rapporto_purezza - soglia_inizio_laser) / (soglia_laser_pieno - soglia_inizio_laser)
        
    guadagno_disponibile_dB = 5.0 + (k_coer ** 2) * (guadagno_dB - 5.0)
    
    # se P_totale_cavita si avvicina o supera P_sat il guadagno disponibile crolla drasticamente
    guadagno_saturo_dB = guadagno_disponibile_dB / (1.0 + (P_totale_cavita / P_sat))
    
    # COMPETIZIONE TRA I MODI
    if k_coer > 0 and max_potenza > 1e-9:
        distanza_dal_massimo = 1.0 - (potenze_in / max_potenza)
        penalizzazione_modi_dB = 25.0 * (distanza_dal_massimo ** 2)
        guadagno_effettivo_dB = guadagno_saturo_dB - penalizzazione_modi_dB
    else:
        guadagno_effettivo_dB = guadagno_saturo_dB
        
    # Conversione 
    g_lineare_vettore = 10 ** (guadagno_effettivo_dB / 20)
    E_stimolata = E_ingresso_filtrato * g_lineare_vettore
    
    # Rumore di fondo permanente
    rumore_casuale = np.random.normal(0, 0.05, len(lambda_vett)) + 1j * np.random.normal(0, 0.05, len(lambda_vett))
    campana_spontanea = np.exp(-((lambda_vett - 1550e-9) / 40e-9)**2)
    E_ase = rumore_casuale * campana_spontanea
    
    E_out = (1.0 - 0.95 * k_coer) * E_ase + E_stimolata
    
    return E_out
    
def phase_selector_control(L_cavita_totale, lambda_desiderata):

    n_eff_base = 1.7        
    dn_dT = 1.8e-4          
    L_phase = 500e-6        
    
    # Calcoliamo la fase accumulata nel chip a temperatura ambiente (andata e ritorno)
    fase_base = (2 * np.pi / lambda_desiderata) * (2 * n_eff_base * L_cavita_totale)

    fase_residua = fase_base % (2 * np.pi)
    
    # Calcoliamo la fase mancante per arrivare al prossimo multiplo intero successivo di 2*pi
    if fase_residua == 0:
        delta_fase_necessario = 0.0
    else:
        delta_fase_necessario = (2 * np.pi) - fase_residua
        
    # Convertiamo questo sfasamento in variazione di indice
    delta_n_necessario = (delta_fase_necessario * lambda_desiderata) / (4 * np.pi * L_phase)
    
    # Convertiamo l'indice in temperatura tramite il coefficiente
    delta_T_phase = delta_n_necessario / dn_dT
    
    return delta_T_phase

def phase_section(E_in, omega, delta_T_phase):

    c = 3e8
    L_phase = 500e-6
    n_eff_base = 1.7
    dn_dT = 1.8e-4
    
    # Indice modificato dal calore calcolato dal controllo
    n_eff_corrente = n_eff_base + (dn_dT * delta_T_phase)
    
    # Fase accumulata
    phi = (n_eff_corrente * omega / c) * L_phase
    E_out = E_in * np.exp(1j * phi)
    
    return E_out

def tunable_coupler(E_in, K_divisione=0.7):

    # Controllo sul coefficente per non esplodere
    K_divisione = np.clip(K_divisione, 0.0, 1.0)
    
    # Quadro perchè campi e non potenze
    t_campo = np.sqrt(K_divisione)          # Coefficiente di trasmissione verso i ring
    r_campo = np.sqrt(1 - K_divisione)      # Coefficiente di riflessione verso out
    
    E_verso_ring = E_in * t_campo
    E_output_estratto = E_in * r_campo
    
    return E_verso_ring, E_output_estratto

def _simula_singolo_ring(omega, R, n_eff, n_g, kappa, perdite_dB_cm):
    
    c = 3e8
    L = 2 * np.pi * R                     # Circonferenza dell'anello
    omega_0 = 2 * np.pi * (c / 1550e-9)   # Pulsazione di riferimento (1550 nm)
    
    alpha_m = (perdite_dB_cm / 4.34) * 100 
    a = np.exp(-alpha_m * L / 2)          
    t = np.sqrt(1 - kappa)                # Coefficiente di accoppiamento passante
    
    # Approssimazione lineare dell'indice efficace con la dispersione (n_g)
    theta = ((n_eff * omega_0 / c) + (n_g * (omega - omega_0) / c)) * L
    
    # Formula analitica del microring
    numeratore = ((1 - t**2)**2) * a
    denominatore = 1 + (t**4) * (a**2) - 2 * (t**2) * a * np.cos(theta)
    
    return numeratore / denominatore

def specchio_vernier(E_in, omega, T_ring2_C=25.0):
    
    R1 = 100e-6     # 100 micrometri
    R2 = 105e-6     # 105 micrometri
    
    # Parametri ottici fisici delle guide
    n_eff_base = 1.7
    n_g = 2.0
    kappa = 0.22    
    perdite = 1.0          # dB/cm   
    dn_dT = 1.8e-5  
    
    # RING 1: Rimane fisso a temperatura ambiente
    T1_potenza = _simula_singolo_ring(omega, R1, n_eff_base, n_g, kappa, perdite)
    
    # RING 2: Viene scaldato per fare il tuning del laser
    delta_T = T_ring2_C - 25.0
    n_eff_riscaldato = n_eff_base + (dn_dT * delta_T)
    T2_potenza = _simula_singolo_ring(omega, R2, n_eff_riscaldato, n_g, kappa, perdite)
    
    # Risposta combinata
    T_vernier_totale = (T1_potenza * T2_potenza) ** 4
    
    # Il campo elettrico riflesso (sqrt potenza)
    E_riflesso = E_in * np.sqrt(T_vernier_totale)
    
    return E_riflesso
    
    # SOA
    E_soa = SOA_InP_auto_coerente(E_in_cavita, lambda_vett, guadagno_dB=24)
    
    # Phase section
    E_fase = phase_section(E_soa, omega_vett, delta_T_phase=T_fase)
    
    # tunable coupler
    E_verso_ring, E_estratto_fuori = tunable_coupler(E_fase, K_divisione=K_div)
    
    # ring
    E_riflesso_da_ring = specchio_vernier(E_verso_ring, omega_vett, T_ring2_C=T_r2)
    
    return E_riflesso_da_ring, E_estratto_fuori

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import io
from PIL import Image

if __name__ == "__main__":
    
    lambda_vett = np.linspace(1530e-9, 1570e-9, 3000)
    omega_vett = 2 * np.pi * 3e8 / lambda_vett
    x_nm = lambda_vett * 1e9
    
    L_cavita_reale = 5.002e-3     # 5 mm + tolleranza geometrica
    lambda_desiderata = 1550e-9   # Canale target del filtro
    K_divisione_MZI = 0.75        # 75% potenza dentro la cavità, 25% estratta
    T_ring2_scaldato = 55.0       # Temperatura del secondo anello
    
    # Il controllore di fase imposta il riscaldatore prima dell'accensione del laser
    T_fase_ottima = phase_selector_control(L_cavita_reale, lambda_desiderata)
    print(f"[Phase Control]: Riscaldatore impostato staticamente a {T_fase_ottima:.2f} °C")
    
    # Inizializziamo il campo elettrico della cavità a zero
    E_cavita_corrente = np.zeros_like(lambda_vett) + 1j*0  # Al tempo zero la cavità è vuota
    num_passaggi = 35  # Numero di passaggi totali per osservare la stabilizzazione
    
    lista_frame_immagini = []
    print("\nGenerazione della GIF")
    plt.ioff()
    
    for frame in range(num_passaggi):
        
        E_soa = SOA_InP_auto_coerente(E_cavita_corrente, lambda_vett, guadagno_dB=24.0)
        
        E_fase = phase_section(E_soa, omega_vett, delta_T_phase=T_fase_ottima)
        
        E_verso_ring, E_uscita_istantanea = tunable_coupler(E_fase, K_divisione=K_divisione_MZI)
        
        E_cavita_corrente = specchio_vernier(E_verso_ring, omega_vett, T_ring2_C=T_ring2_scaldato)
        
        # Grafico
        # to dBm
        P_cavita_dBm = 10 * np.log10(np.abs(E_verso_ring)**2 + 1e-12)       # Sonda porta interna
        P_uscita_dBm = 10 * np.log10(np.abs(E_uscita_istantanea)**2 + 1e-12) # Sonda porta esterna
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"Evoluzione Temporale del Laser - Rimbalzo Ottico: {frame + 1}/{num_passaggi}", 
                     fontsize=12, fontweight='bold')
        

        limite_y_inf = -100 
        limite_y_sup = 40 
        
        ax1.plot(x_nm, P_cavita_dBm, color="teal", linewidth=1.5, label="Internal Cavity")
        ax1.set_xlim(1530, 1570)
        ax1.set_ylim(limite_y_inf, limite_y_sup)
        ax1.set_xlabel("Wavelength (nm)", fontweight='bold')
        ax1.set_ylabel("Internal Power (dBm)", fontweight='bold')
        ax1.grid(True, linestyle=":", alpha=0.6)
        ax1.legend(loc="upper right")
        
        ax2.plot(x_nm, P_uscita_dBm, color="crimson", linewidth=1.5, label="Laser Output")
        ax2.set_xlim(1530, 1570)
        ax2.set_ylim(limite_y_inf, limite_y_sup)
        ax2.set_xlabel("Wavelength (nm)", fontweight='bold')
        ax2.set_ylabel("Output Power (dBm)", fontweight='bold')
        ax2.grid(True, linestyle=":", alpha=0.6)
        ax2.legend(loc="upper right")
        
        plt.tight_layout()
        
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100)
        buf.seek(0)
        img_frame = Image.open(buf)
        img_frame.load()
        lista_frame_immagini.append(img_frame)
        plt.close(fig)
        
    plt.ion()
    
    # compilazione finale della GIF
    if lista_frame_immagini:
        nome_file_gif = "evoluzione_laser_auto_coerente.gif"
        print(f"\n[Pillow] Generazione del file GIF in corso...")
        lista_frame_immagini[0].save(
            nome_file_gif,
            save_all=True,
            append_images=lista_frame_immagini[1:],
            duration=220,  # ms
            loop=0
        )
        print(f"[Pillow] Animazione completata con successo: '{nome_file_gif}'")

# Grafico Saturazione


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Image, display

guadagno_nominal_dB = 20.0
P_sat = 1.0  # Barriera di saturazione a 1 Watt
P_tot_vett = np.linspace(0, 5, 200) # Asse X: Potenza da 0 a 5 Watt

# Calcolo della curva statica di base (k_coer = 1.0 costante per la barriera)
g_disp_max = 5.0 + (1.0 ** 2) * (guadagno_nominal_dB - 5.0)
g_sat_curva_completa = g_disp_max / (1.0 + (P_tot_vett / P_sat))

fig, ax = plt.subplots(figsize=(7, 5))

# Funzione di inizializzazione del grafico
def init():
    ax.clear()
    ax.axvline(P_sat, color='black', linestyle='--', alpha=0.7, label=f'P_sat ({P_sat} W)')
    ax.set_title("Dinamica di Saturazione del Guadagno", fontsize=11, fontweight='bold')
    ax.set_xlabel("Potenza Totale in Cavità (W)", fontsize=10)
    ax.set_ylabel("Guadagno al Picco (dB)", fontsize=10)
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 22)
    ax.grid(True, linestyle=':', alpha=0.6)
    return []

def update(frame):
    init() # Ridisegna la base fissa
    
    P_attuale = (frame / 40) * 4.5 
    
    # Disegna la porzione di curva percorsa fino a questo momento
    P_percorso = np.linspace(0, P_attuale, 100)
    G_percorso = g_disp_max / (1.0 + (P_percorso / P_sat))
    ax.plot(P_percorso, G_percorso, color='crimson', linewidth=3)
        
    ax.legend(loc='upper left', fontsize=8)
    return []

ani = animation.FuncAnimation(fig, update, frames=40, init_func=init, blit=True)

# Salva l'animazione come file GIF
gif_filename = 'saturazione_guadagno_barriera.gif'
ani.save(gif_filename, writer='pillow', fps=10)
plt.close()

display(Image(filename=gif_filename))

# Rumore -> Laser

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


lambda_vett = np.linspace(1530e-9, 1570e-9, 3000)
omega_vett = 2 * np.pi * 3e8 / lambda_vett
x_nm = lambda_vett * 1e9

L_cavita_reale = 5.002e-3     # 5 mm + tolleranza geometrica
lambda_desiderata = 1550e-9   # Canale target del filtro
K_divisione_MZI = 0.75        # 75% potenza dentro la cavità, 25% estratta
T_ring2_scaldato = 55.0       # Temperatura del secondo anello

# Il controllore di fase imposta il riscaldatore prima dell'accensione del laser
T_fase_ottima = phase_selector_control(L_cavita_reale, lambda_desiderata)
print(f"[Phase Control]: Riscaldatore impostato staticamente a {T_fase_ottima:.2f} °C")

# Inizializziamo il campo elettrico della cavità a zero
E_cavita_corrente = np.zeros_like(lambda_vett) + 1j*0  # Al tempo zero la cavità è vuota

num_passaggi = 1  # Numero di passaggi totali per osservare la stabilizzazione

for frame in range(num_passaggi):

    E_soa = SOA_InP_auto_coerente(E_cavita_corrente, lambda_vett, guadagno_dB=24.0)
            
    E_fase = phase_section(E_soa, omega_vett, delta_T_phase=T_fase_ottima)

    E_verso_ring, E_uscita_istantanea = tunable_coupler(E_fase, K_divisione=K_divisione_MZI)

    E_cavita_corrente = specchio_vernier(E_verso_ring, omega_vett, T_ring2_C=T_ring2_scaldato)

# Grafico
# to dBm
P_uscita_dBm = 10 * np.log10(np.abs(E_uscita_istantanea)**2 + 1e-12) # Sonda porta esterna


#Grafico
plt.figure(figsize=(10, 5))
plt.plot(lambda_vett * 1e9, P_uscita_dBm, color='orange', linewidth=1.5, label='$E_{out}$')
plt.title("Passaggio da rumore a laser", fontsize=12, fontweight='bold')
plt.xlabel("Lunghezza d'onda (nm)", fontsize=10)
plt.ylabel("Potenza Spettrale (dBm)", fontsize=10)
plt.xlim(1530, 1570)
plt.ylim(-45, -5)
plt.grid(True, linestyle=':', alpha=0.6)

plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Ripartizione potenza

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambda_vett = np.linspace(1548e-9, 1552e-9, 1000) # Zoom attorno al picco

# Simuliamo un campo elettrico complesso in ingresso al coupler (Picco + Rumore di fondo)
np.random.seed(24)
rumore = np.random.normal(0, 0.02, len(lambda_vett)) + 1j * np.random.normal(0, 0.02, len(lambda_vett))
picco_laser = np.exp(-((lambda_vett - 1550e-9) / 0.1e-9)**2) * 5.0 # Picco stretto e potente
E_in = picco_laser + rumore

P_in = np.abs(E_in) ** 2

K_divisione = 0.75

E_verso_ring = E_in * np.sqrt(K_divisione)
E_uscita = E_in * np.sqrt(1.0 - K_divisione)

P_verso_ring = np.abs(E_verso_ring) ** 2
P_uscita = np.abs(E_uscita) ** 2

plt.figure(figsize=(11, 5))

# Plot della potenza trattenuta in cavità (75%)
plt.plot(lambda_vett * 1e9, P_verso_ring, color='royalblue', linewidth=2.5, 
         label=f'Alimentazione Cavità / Ring ({K_divisione*100:.0f}%)')

# Plot della potenza estratta dal chip (25%)
plt.plot(lambda_vett * 1e9, P_uscita, color='limegreen', linewidth=2, 
         label=f'Uscita Laser Utile ({(1.0 - K_divisione)*100:.0f}%)')

# Linea tratteggiata per la potenza totale di riferimento
plt.plot(lambda_vett * 1e9, P_in, color='black', linestyle=':', alpha=0.5, label='Potenza Totale in Ingresso ($E_{in}$)')

# Configurazione estetica
plt.title("Ripartizione della Potenza nel Tunable Coupler ($K_{div} = 0.75$)", fontsize=12, fontweight='bold')
plt.xlabel("Lunghezza d'onda (nm)", fontsize=10)
plt.ylabel("Potenza Lineare (u.a.)", fontsize=10)
plt.xlim(1549, 1551)
plt.grid(True, linestyle=':', alpha=0.6)

plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Filtraggio Microring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambda_vett = np.linspace(1530e-9, 1570e-9, 20000) # Alta risoluzione per non perdere i picchi stretti
c = 3e8
omega_vett = 2 * np.pi * c / lambda_vett

# Generiamo il rumore ASE di base
np.random.seed(42)
rumore_casuale = np.random.normal(0, 0.05, len(lambda_vett)) + 1j * np.random.normal(0, 0.05, len(lambda_vett))
campana_spontanea = np.exp(-((lambda_vett - 1550e-9) / 25e-9)**2)
E_in_ase = rumore_casuale * campana_spontanea

# Lasciamo T_ring2_C = 25.0 (temperatura ambiente) per vedere dove si allineano i picchi naturali.
E_dopo_vernier = specchio_vernier(E_in_ase, omega_vett, T_ring2_C=25.0)

P_prima = np.abs(E_in_ase) ** 2
P_dopo = np.abs(E_dopo_vernier) ** 2

P_prima_dB = 10 * np.log10(P_prima + 1e-18)
P_dopo_dB = 10 * np.log10(P_dopo + 1e-18)

plt.figure(figsize=(12, 5))

# rumore di ingresso (Grigio)
plt.plot(lambda_vett * 1e9, P_prima_dB, color='lightgray', alpha=0.7, label='Prima: Rumore Spontaneo')

# output
plt.plot(lambda_vett * 1e9, P_dopo_dB, color='darkorchid', linewidth=1.2, label='Dopo: Sprettro Filtrato')

# Configurazione grafica
plt.title("Spettro prima/dopo il filtraggio dei microring", fontsize=12, fontweight='bold')
plt.xlabel("Lunghezza d'onda (nm)", fontsize=10)
plt.ylabel("Potenza Spettrale (dB)", fontsize=10)
plt.xlim(1530, 1570)
plt.ylim(-120, 10) 
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Confronto perdite/larghezza di riga

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_cicli = 35
vettore_perdite = [0.5, 2.0, 5.0, 12.0, 20.0]  # dB/cm
colori = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

matrice_fwhm = np.zeros((len(vettore_perdite), num_cicli))

for p_idx, loss in enumerate(vettore_perdite):
    # Parametri geometrici dei ring
    R1, R2 = 100e-6, 105e-6
    L_giro = 2 * np.pi * (R1 + R2)
    kappa = 0.22
    t = np.sqrt(1 - kappa)
    
    # Coefficiente di perdita lineare per singolo giro
    alpha_m = (loss / 4.34) * 100  # in 1/m
    a_giro = np.exp(-alpha_m * L_giro / 2)

    fwhm_strutturale = 0.02 * (1.0 + (1.0 - a_giro) * 15.0)  # in nm
    
    for ciclo in range(num_cicli):
        # Transitorio di accensione: il laser stringe lo spettro ciclo dopo ciclo
        fattore_tempo = 1.0 / np.sqrt(ciclo + 1)
        
        # Nel chip ottimo (0.5 dB/cm), la riga crolla verso lo zero.
        # Nel chip a 20 dB/cm, il limite inferiore strutturale impedisce il restringimento.
        fwhm_dinamica = 0.6 * fattore_tempo + fwhm_strutturale
        
        # Saturazione fisica al limite del filtro
        if fwhm_dinamica < fwhm_strutturale * 1.1:
            fwhm_dinamica = fwhm_strutturale * 1.1
            
        matrice_fwhm[p_idx, ciclo] = fwhm_dinamica

fig, ax = plt.subplots(figsize=(10, 6))
x_cicli = np.arange(1, num_cicli + 1)

for p_idx, loss in enumerate(vettore_perdite):
    ax.plot(x_cicli, matrice_fwhm[p_idx, :], 
            color=colori[p_idx], 
            linewidth=2.5, 
            label=f"Perdita: {loss} dB/cm")

ax.set_xlim(1, num_cicli)
ax.set_ylim(0.0, 0.7) 

ax.set_xlabel("Cicli di Cavità (Iterazioni del Main Loop)", fontsize=11)
ax.set_ylabel("Larghezza di riga del Laser FWHM (nm)", fontsize=11)
ax.set_title("Evoluzione della FWHM: Confronto tra Chip Ottimizzati e Degradati", fontsize=12, fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='upper right', fontsize=10)

# Salva e mostra
plt.savefig('risultato_tesi_funzionante.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambda_vett = np.linspace(1549.6e-9, 1550.4e-9, 5000)
lambda_nm = lambda_vett * 1e9
c = 3e8
omega_vett = 2 * np.pi * c / lambda_vett

vettore_perdite = [0.5, 2.0, 5.0, 12.0, 20.0] # dB/cm
colori = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
num_cicli = 35
K_divisione = 0.75

shift_incrementale = 0.2

np.random.seed(42)
E_in_base = (np.random.normal(0, 0.02, len(lambda_vett)) + 1j * np.random.normal(0, 0.02, len(lambda_vett)))

def _risposta_ring(omega, R, kappa, perdite_dB_cm):
    L = 2 * np.pi * R
    alpha_m = (perdite_dB_cm / 4.34) * 100 
    a = np.exp(-alpha_m * L / 2)          
    t = np.sqrt(1 - kappa)
    omega_0 = 2 * np.pi * (c / 1550e-9)
    theta = (2.0 * (omega - omega_0) / c) * L
    numeratore = ((1 - t**2)**2) * a
    denominatore = 1 + (t**4) * (a**2) - 2 * (t**2) * a * np.cos(theta)
    return numeratore / denominatore

plt.figure(figsize=(10, 6))

for p_idx, loss in enumerate(vettore_perdite):
    E_cavita = np.copy(E_in_base)
    
    T1 = _risposta_ring(omega_vett, 100e-6, 0.22, loss)
    T2 = _risposta_ring(omega_vett, 105e-6, 0.22, loss)
    T_vernier_tot = T1 * T2
    
    for ciclo in range(num_cicli):
        P_media = np.mean(np.abs(E_cavita)**2)
        G = 24.0 / (1.0 + P_media * 0.8)
        E_cavita = E_cavita * np.sqrt(10**(G/10))
        E_cavita = E_cavita * np.sqrt(T_vernier_tot)
        
        E_uscita = E_cavita * np.sqrt(1.0 - K_divisione)
        E_cavita = E_cavita * np.sqrt(K_divisione)
    
    Spettro_Potenza_Reale = np.abs(E_uscita)**2

    Spettro_dB = 10 * np.log10(Spettro_Potenza_Reale + 1e-15)
    
    lambda_shifted = lambda_nm + (p_idx * shift_incrementale)
    
    plt.plot(lambda_shifted, Spettro_dB, 
             color=colori[p_idx], 
             linewidth=2.0, 
             label=f"Perdita: {loss} dB/cm")

plt.xlim(1549.7, 1551.2) 
plt.ylim(-80, 50) 

plt.xlabel("Lunghezza d'onda (nm) + Shift artificiale", fontsize=11)
plt.ylabel("Potenza Spettrale d'Uscita (dB)", fontsize=11)
plt.title(f"Spettro della Potenza di Cavità al {num_cicli}° Ciclo", fontsize=12, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', fontsize=10)

# Salva e mostra
plt.savefig('spettri_potenza_dB.png', dpi=300, bbox_inches='tight')
plt.show()

# Confrotnto con k

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lambda_vett = np.linspace(1549.6e-9, 1550.4e-9, 5000)
lambda_nm = lambda_vett * 1e9
c = 3e8
omega_vett = 2 * np.pi * c / lambda_vett

vettore_kappa = [0.05, 0.12, 0.22, 0.35, 0.45] 
colori = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
num_cicli = 35
K_divisione = 0.75
perdita_fissa = 0.5 # Perdita ottimale fissata a 0.5 dB/cm

shift_incrementale = 0.1 

np.random.seed(42)
E_in_base = (np.random.normal(0, 0.02, len(lambda_vett)) + 1j * np.random.normal(0, 0.02, len(lambda_vett)))

def _risposta_ring(omega, R, kappa_variabile, perdite_dB_cm):
    L = 2 * np.pi * R
    alpha_m = (perdite_dB_cm / 4.34) * 100 
    a = np.exp(-alpha_m * L / 2)          
    t = np.sqrt(1 - kappa_variabile) # t dipende dal kappa corrente
    omega_0 = 2 * np.pi * (c / 1550e-9)
    theta = (2.0 * (omega - omega_0) / c) * L
    numeratore = ((1 - t**2)**2) * a
    denominatore = 1 + (t**4) * (a**2) - 2 * (t**2) * a * np.cos(theta)
    return numeratore / denominatore

plt.figure(figsize=(10, 6))

for k_idx, k_val in enumerate(vettore_kappa):
    E_cavita = np.copy(E_in_base)
    
    T1 = _risposta_ring(omega_vett, 100e-6, k_val, perdita_fissa)
    T2 = _risposta_ring(omega_vett, 105e-6, k_val, perdita_fissa)
    T_vernier_tot = T1 * T2
    
    for ciclo in range(num_cicli):
        P_media = np.mean(np.abs(E_cavita)**2)
        G = 24.0 / (1.0 + P_media * 0.8)
        E_cavita = E_cavita * np.sqrt(10**(G/10))
        E_cavita = E_cavita * np.sqrt(T_vernier_tot)
        
        E_uscita = E_cavita * np.sqrt(1.0 - K_divisione)
        E_cavita = E_cavita * np.sqrt(K_divisione)
    
    Spettro_Potenza_Reale = np.abs(E_uscita)**2
    Spettro_dB = 10 * np.log10(Spettro_Potenza_Reale + 1e-15)
    
    lambda_shifted = lambda_nm + (k_idx * shift_incrementale)
    
    plt.plot(lambda_shifted, Spettro_dB, 
             color=colori[k_idx], 
             linewidth=2.0, 
             label=f"Accoppiamento K: {k_val}")

plt.xlim(1549.7, 1550.7) 
plt.ylim(-80, 50) 

plt.xlabel("Lunghezza d'onda (nm) + Shift artificiale", fontsize=11)
plt.ylabel("Potenza Spettrale d'Uscita (dB)", fontsize=11)
plt.title(f"Analisi di Tolleranza: Spettro al {num_cicli}° Ciclo variando il fattore K", fontsize=12, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=10)

# Salva e mostra
plt.savefig('spettri_variazione_K_dB.png', dpi=300, bbox_inches='tight')
plt.show()

# Grafico 3D


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

lambda_vett = np.linspace(1549.6e-9, 1550.4e-9, 2000)
c = 3e8
omega_vett = 2 * np.pi * c / lambda_vett
num_cicli = 35
K_divisione = 0.75

np.random.seed(42)
E_in_base = (np.random.normal(0, 0.02, len(lambda_vett)) + 1j * np.random.normal(0, 0.02, len(lambda_vett)))

def _risposta_ring(omega, R, kappa, perdite_dB_cm):
    L = 2 * np.pi * R
    alpha_m = (perdite_dB_cm / 4.34) * 100 
    a = np.exp(-alpha_m * L / 2)          
    t = np.sqrt(1 - kappa)
    omega_0 = 2 * np.pi * (c / 1550e-9)
    theta = (2.0 * (omega - omega_0) / c) * L
    numeratore = ((1 - t**2)**2) * a
    denominatore = 1 + (t**4) * (a**2) - 2 * (t**2) * a * np.cos(theta)
    return numeratore / denominatore

def _simula_cavita(loss_val, kappa_val):
    E_cavita = np.copy(E_in_base)
    T1 = _risposta_ring(omega_vett, 100e-6, kappa_val, loss_val)
    T2 = _risposta_ring(omega_vett, 105e-6, kappa_val, loss_val)
    T_vernier_tot = T1 * T2
    
    for ciclo in range(num_cicli):
        P_media = np.mean(np.abs(E_cavita)**2)
        G = 24.0 / (1.0 + P_media * 0.8)
        E_cavita = E_cavita * np.sqrt(10**(G/10))
        E_cavita = E_cavita * np.sqrt(T_vernier_tot)
        E_uscita = E_cavita * np.sqrt(1.0 - K_divisione)
        E_cavita = E_cavita * np.sqrt(K_divisione)
    return np.abs(E_uscita)**2

P_riferimento = _simula_cavita(loss_val=0.5, kappa_val=0.22)

Asse_Perdite = np.linspace(0.5, 15.0, 20)  # da 0.5 a 15 dB/cm
Asse_Kappa = np.linspace(0.05, 0.45, 20)    # da 0.05 a 0.45
X_loss, Y_kappa = np.meshgrid(Asse_Perdite, Asse_Kappa)
Z_mse = np.zeros_like(X_loss)

for i in range(X_loss.shape[0]):
    for j in range(X_loss.shape[1]):
        l_corrente = X_loss[i, j]
        k_corrente = Y_kappa[i, j]
        
        # Simula il caso di test
        P_test = _simula_cavita(loss_val=l_corrente, kappa_val=k_corrente)
        
        # Calcolo MSE rispetto al riferimento nominale
        Z_mse[i, j] = np.mean((P_riferimento - P_test)**2)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

superficie = ax.plot_surface(X_loss, Y_kappa, Z_mse, 
                             cmap='viridis', 
                             edgecolor='none', 
                             alpha=0.9)

cbar = fig.colorbar(superficie, ax=ax, shrink=0.5, aspect=10)
cbar.set_label('Mean Squared Error (MSE)', fontsize=10)

# Etichette degli assi
ax.set_xlabel('Perdite di Propagazione (dB/cm)', fontsize=11, labelpad=10)
ax.set_ylabel('Coefficiente di Accoppiamento K', fontsize=11, labelpad=10)
ax.set_zlabel('MSE dello Spettro', fontsize=11, labelpad=10)
ax.set_title('Superficie di Errore 3D: Sensibilità del Laser a K e Perdite', fontsize=13, fontweight='bold')

ax.view_init(elev=30, azim=220)

plt.savefig('mappa_errore_3D_tesi.png', dpi=300, bbox_inches='tight')
plt.show()